# ch04 练习：回归模型

与讲解版逐 Cell 对应。核心代码被挖空，`assert` 验收**保留**——
跑通所有 `assert` 即自查通过。卡住就回讲解版看对应小节。

## 一、考点回顾

| 竞赛评分点 | 本节覆盖 |
|---|---|
| 模型训练 40% | 线性 / Ridge / Lasso / DT / RF / GBDT 的 fit-predict |
| 模型性能评估 10% | R² / RMSE / MAE 与负 R² 陷阱 |

## 二、API 速查

| 类 | 关键参数 | 一句话 |
|---|---|---|
| `LinearRegression` | — | 最小二乘基线 |
| `Ridge(alpha)` | L2 收缩 | 只压不删 |
| `Lasso(alpha)` | L1 归零 | 尺度敏感，先标准化 |
| `DecisionTreeRegressor(max_depth)` | 剪枝阀门 | 不限深=背表 |
| `RandomForestRegressor(n_estimators)` | Bagging | importance 可读 |
| `GradientBoostingRegressor` | Boosting | 本数据最强 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

import sklearn
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

DATA = Path("data")

reg = pd.read_csv(DATA / "load_reg.csv")     # 回归：预测「日均负荷」

NUM = ["容量kVA", "投运年限", "月平均温度", "月平均湿度", "户数"]
CAT = ["供电区域类型"]
TARGET = "日均负荷"

# 台区编号是 ID 列，直接当特征会出事（§3.7 演示），本章特征矩阵不含它
tr_idx, te_idx = train_test_split(np.arange(len(reg)), test_size=0.2, random_state=42)
reg_tr, reg_te = reg.iloc[tr_idx], reg.iloc[te_idx]
y_tr, y_te = reg_tr[TARGET], reg_te[TARGET]

# 独热 + 标准化（预处理细节 ch02 讲透，这里直接组装）
oh = OneHotEncoder(sparse_output=False).fit(reg_tr[CAT])
feat = NUM + [f"区域_{c}" for c in oh.categories_[0]]
Xtr = np.hstack([reg_tr[NUM].to_numpy(), oh.transform(reg_tr[CAT])])
Xte = np.hstack([reg_te[NUM].to_numpy(), oh.transform(reg_te[CAT])])
ss = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = ss.transform(Xtr), ss.transform(Xte)

print("sklearn", sklearn.__version__)
print("train", Xtr_s.shape, "test", Xte_s.shape,
      "| y 均值 train", round(float(y_tr.mean()), 2), "test", round(float(y_te.mean()), 2))

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def reg_report(tag, y_true, y_pred):
    """一行输出 R2 / RMSE / MAE，返回 (r2, rmse, mae)。"""
    r2 = r2_score(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = mean_absolute_error(y_true, y_pred)
    print(f"{tag}: R2={r2:.4f} RMSE={rmse:.4f} MAE={mae:.4f}")
    return r2, rmse, mae


print("脚手架就绪：reg_report")

## 题 1：线性回归与两种尺度（对应讲解 §3.1-3.2）

In [ ]:
lin = LinearRegression().fit(Xtr_s, y_tr)
lin_raw = LinearRegression().fit(Xtr, y_tr)

print("scaled coef_:", np.round(lin.coef_, 4).tolist())
print("scaled intercept:", round(float(lin.intercept_), 4))
print("raw coef_   :", np.round(lin_raw.coef_, 4).tolist())

reg_report("LIN  train", y_tr, lin.predict(Xtr_s))
r2_te, rmse_te, mae_te = reg_report("LIN  test ", y_te, lin.predict(Xte_s))
reg_report("LIN-raw test", y_te, lin_raw.predict(Xte))

top = feat[int(np.argmax(np.abs(lin.coef_[: len(NUM)])))]
print("数值特征中 |系数| 最大:", top)

assert round(float(lin.intercept_), 4) == 393.0875
assert round(r2_te, 4) == 0.9234 and round(rmse_te, 4) == 48.3232
assert round(float(mae_te), 4) == 37.6304
assert round(r2_score(y_tr, lin.predict(Xtr_s)), 4) == 0.9480
assert round(r2_score(y_te, lin_raw.predict(Xte)), 4) == 0.9234, "R2 对线性缩放不变"
assert not np.allclose(lin.coef_, lin_raw.coef_), "但系数完全不同"
assert top == "容量kVA", "线性系数视角: 容量最强(113.57)"

**为什么这样做**：scaled 与 raw 各 fit 一次不是浪费——R² 的一致性证明
「缩放不影响预测」，系数的差异证明「缩放影响解读」。两者合起来才是完整结论。

## 题 2：Ridge 收缩与 Lasso 归零（对应讲解 §3.3）

In [ ]:
rd10 = Ridge(alpha=10.0).fit(Xtr_s, y_tr)
r2_ridge = r2_score(y_te, rd10.predict(Xte_s))
print("Ridge a=10 test R2:", round(r2_ridge, 4),
      "| 容量系数:", round(float(rd10.coef_[0]), 3))

la1 = Lasso(alpha=1.0).fit(Xtr_s, y_tr)
la10 = Lasso(alpha=10.0).fit(Xtr_s, y_tr)
nz1, nz10 = int((la1.coef_ != 0).sum()), int((la10.coef_ != 0).sum())
kept10 = {feat[i] for i in range(len(feat)) if la10.coef_[i] != 0}
print("Lasso a=1 非零:", nz1, "/9 | a=10 非零:", nz10, "/9 | a=10 保留:", kept10)

assert round(r2_ridge, 4) == 0.9245
assert round(float(rd10.coef_[0]), 3) == 102.09, "Ridge 只收缩不归零"
assert nz1 == 6 and nz10 == 3
assert kept10 == {"容量kVA", "户数", "区域_工业区"}, "a=10 只留最强三路信号"

## 题 3：Lasso 的尺度敏感性（对应讲解 §3.4）

In [ ]:
la_s = Lasso(alpha=1.0).fit(Xtr_s, y_tr)
la_r = Lasso(alpha=1.0).fit(Xtr, y_tr)
nz_s, nz_r = int((la_s.coef_ != 0).sum()), int((la_r.coef_ != 0).sum())
print("scaled 非零:", nz_s, "| raw 非零:", nz_r)
print("scaled coef:", np.round(la_s.coef_, 3).tolist())
print("raw   coef:", np.round(la_r.coef_, 4).tolist(),
      "← 容量kVA 量纲大(百位)，原始尺度下惩罚相对变小")

assert nz_r == 7
assert round(float(la_r.coef_[0]), 4) == 0.3892, "raw 版容量系数 0.39 vs scaled 版 113"
assert round(float(abs(la_s.coef_[0])), 3) == 113.061, "同一特征，两种尺度差 290 倍"

## 题 4：决策树的背表与剪枝（对应讲解 §3.5）

In [ ]:
dt = DecisionTreeRegressor(random_state=42).fit(Xtr_s, y_tr)
reg_report("DT-all train", y_tr, dt.predict(Xtr_s))
r2_dt = r2_score(y_te, dt.predict(Xte_s))
print("全深树: depth =", dt.get_depth(), "| leaves =", dt.get_n_leaves())
print("DT-all test R2:", round(r2_dt, 4))

dt9 = DecisionTreeRegressor(max_depth=9, random_state=42).fit(Xtr_s, y_tr)
r2_d9 = r2_score(y_te, dt9.predict(Xte_s))
print("DT-d9 test R2:", round(r2_d9, 4), "← 适度剪枝反而更好")

# 树不吃 scale：逐列线性变换不改变「谁和谁比较」
dt_ns = DecisionTreeRegressor(max_depth=5, random_state=42).fit(Xtr, y_tr)
dt5 = DecisionTreeRegressor(max_depth=5, random_state=42).fit(Xtr_s, y_tr)
print("树 no-scale 与 scale 版预测逐位一致:",
      bool((dt_ns.predict(Xtr) == dt5.predict(Xtr_s)).all()))

assert dt.get_depth() == 21 and dt.get_n_leaves() == 640, "640 行被切到 640 片叶子"
assert round(r2_score(y_tr, dt.predict(Xtr_s)), 4) == 1.0, "全深树背下训练段"
assert round(r2_dt, 4) == 0.9228
assert round(r2_d9, 4) == 0.9339, "适度剪枝反而更好"
assert (dt_ns.predict(Xtr) == dt5.predict(Xtr_s)).all()

## 题 5：随机森林与 GBDT（对应讲解 §3.6）

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=1).fit(Xtr_s, y_tr)
rf_r2 = r2_score(y_te, rf.predict(Xte_s))

gbr = GradientBoostingRegressor(random_state=42).fit(Xtr_s, y_tr)
gbr_r2 = r2_score(y_te, gbr.predict(Xte_s))

imp = dict(zip(feat, np.round(rf.feature_importances_, 4).tolist()))
print("RF test R2:", round(rf_r2, 4), "| GBR test R2:", round(gbr_r2, 4))
print("RF 特征重要性:", imp)
print("重要性前 2:", sorted(imp, key=imp.get, reverse=True)[:2])

assert round(rf_r2, 4) == 0.9529 and round(gbr_r2, 4) == 0.9627
assert imp["户数"] == 0.7587 and imp["容量kVA"] == 0.1903
assert sorted(imp, key=imp.get, reverse=True)[:2] == ["户数", "容量kVA"]

## 题 6：负 R² 与 RMSE 的放大效应（对应讲解 §3.7）

In [ ]:
dummy = np.full(len(y_te), y_tr.mean())
r2_dummy = r2_score(y_te, dummy)
print("预测常数=训练均值:", round(r2_dummy, 4), "← 测试段均值略高，R2 变负")

res = np.array([1.0, 1.0, 1.0, 1.0, 10.0])
rmse_r = float(np.sqrt(np.mean(res ** 2)))
mae_r = float(np.mean(res))
print("残差", res.tolist(), "→ RMSE", round(rmse_r, 4), "vs MAE", round(mae_r, 4))

assert round(r2_dummy, 4) == -0.0123, "R2 是可为负的"
assert round(rmse_r, 4) == 4.5607
assert round(mae_r, 4) == 2.8

## 题 7：ID 列陷阱（对应讲解 §3.7）

In [ ]:
oh_id = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(reg_tr[["台区编号"]])
new_row = pd.DataFrame({"台区编号": ["STATION_NEW_99"]})
out_new = oh_id.transform(new_row)
print("训练段学到", oh_id.categories_[0].shape[0], "个台区；测试段独有台区:",
      len(set(reg_te["台区编号"]) - set(reg_tr["台区编号"])), "（这次是运气好）")
print("新台区 STATION_NEW_99 独热后:", out_new.shape,
      "| 该行独热和:", float(out_new.sum()), "← 全 0，模型只能瞎猜")

assert out_new.shape == (1, 36)
assert float(out_new.sum()) == 0.0

---

## 综合自查

1. 题 1 里 R² 一致但系数差 290 倍，各自说明什么？
2. 题 2 的 Lasso a=10 只留三路信号——这三路为什么是它们？
3. 题 3 的 raw 版 Lasso 归零决策为什么不可信？
4. 题 4 全深树 train R²=1.0 而 test 0.9228；d9 反而 0.9339——剪枝在 trade 什么？
5. 题 6 的 −0.0123 是怎么来的？评委问「R2 最低多少」你怎么答？

全部答得上来，进入 ch05（分类模型）。